> **Environment:** This notebook was developed and executed in **Google Colab**.
> It requires external datasets and environment features not available locally.
> Cell outputs are preserved from the original run for reference.

# EcoMarineAI — Быстрый старт в Google Colab

**Время:** 5 минут. **Предварительные знания:** не требуются.

Этот ноутбук показывает как за пять минут:

1. Запустить backend-сервис EcoMarineAI в Colab (Docker не нужен).
2. Отправить изображение кита на идентификацию.
3. Получить вид, индивидуальный идентификатор и уверенность модели.

Если что-то не работает — напишите issue в репозитории: <https://github.com/0x0000dead/whales-identification/issues>.

---

## Шаг 1 — Клонируем репозиторий и ставим зависимости

Колаб даёт ~12 ГБ RAM и 100 ГБ диска — нам хватит с запасом. Модель EfficientNet-B4 весит ~300 МБ, скачается автоматически при первом запуске.

In [ ]:
!git clone --depth 1 https://github.com/0x0000dead/whales-identification.git
%cd whales-identification
!pip install --quiet 'huggingface_hub==0.20.3' 'torch==2.4.1' 'torchvision' 'timm==1.0.11' \
    'fastapi' 'uvicorn[standard]' 'Pillow' 'opencv-python-headless' 'albumentations' \
    'rembg' 'onnxruntime' 'open_clip_torch' 'python-multipart'
print('✓ Зависимости установлены')

## Шаг 2 — Скачиваем веса модели с Hugging Face

Скрипт `scripts/download_models.sh` проверяет SHA256 после загрузки и делает до 3 попыток при сетевых ошибках. Вы можете пропустить проверку контрольных сумм для быстрого старта через `SKIP_CHECKSUMS=1`.

In [ ]:
!SKIP_CHECKSUMS=1 bash scripts/download_models.sh 2>&1 | tail -15

## Шаг 3 — Запускаем FastAPI backend в фоне

В Colab нет Docker, поэтому uvicorn поднимаем напрямую. Сервис занимает ~30 секунд на первый запуск (грузит EfficientNet-B4 в память).

In [ ]:
import subprocess, time, os, sys

os.chdir('whales_be_service')
sys.path.insert(0, 'src')
proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'whales_be_service.main:app',
     '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    env={**os.environ, 'PYTHONPATH': 'src'},
)
print('uvicorn pid:', proc.pid)
# Ждём, пока /health ответит 200 (до 60 секунд).
import urllib.request, urllib.error
for attempt in range(60):
    try:
        r = urllib.request.urlopen('http://localhost:8000/health', timeout=2)
        if r.status == 200:
            print(f'✓ Backend готов за {attempt+1} секунд')
            break
    except (urllib.error.URLError, ConnectionResetError):
        pass
    time.sleep(1)
else:
    raise RuntimeError('Backend не стартовал за 60 секунд')
os.chdir('..')

## Шаг 4 — Загружаем тестовое изображение

В репозитории лежит несколько примеров в `data/datasets/`. Возьмём первый попавшийся.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

candidates = list(Path('data/datasets').glob('*.jpg'))[:1] or list(Path('tests/fixtures').rglob('*.jpg'))[:1]
if not candidates:
    raise SystemExit('Нет примеров в data/datasets/ — загрузите своё фото через вкладку Files слева и укажите путь ниже.')
IMG = candidates[0]
print('Использую:', IMG)
display(Image(filename=str(IMG), width=400))

## Шаг 5 — Отправляем на `/v1/predict-single`

Endpoint принимает multipart/form-data, возвращает JSON с полями:

- `id_animal` — название вида (например `humpback_whale`)
- `class_animal` — индивидуальный ID особи
- `probability` — уверенность модели в идентификации (0.0–1.0)
- `cetacean_score` — уверенность anti-fraud gate (CLIP) что на фото кит или дельфин
- `rejection_reason` — причина если изображение отклонено (`not_a_marine_mammal` / `low_confidence` / `null`)

In [ ]:
import requests, json

with open(IMG, 'rb') as fh:
    r = requests.post(
        'http://localhost:8000/v1/predict-single',
        files={'file': (IMG.name, fh, 'image/jpeg')},
        timeout=30,
    )
print('HTTP', r.status_code)
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

## Шаг 6 — Пакетная обработка (опционально)

Если у вас ZIP с фотографиями от экспедиции, отправьте его на `/v1/predict-batch`:

In [ ]:
# Пример: упаковать 5 файлов в zip и отправить как batch
import zipfile
from pathlib import Path

batch_dir = Path('tmp_batch')
batch_dir.mkdir(exist_ok=True)
zip_path = batch_dir / 'survey.zip'
samples = list(Path('data/datasets').glob('*.jpg'))[:5]
if samples:
    with zipfile.ZipFile(zip_path, 'w') as zf:
        for f in samples:
            zf.write(f, arcname=f.name)
    with open(zip_path, 'rb') as fh:
        r = requests.post(
            'http://localhost:8000/v1/predict-batch',
            files={'archive': ('survey.zip', fh, 'application/zip')},
            timeout=120,
        )
    print('HTTP', r.status_code)
    data = r.json()
    print(f"Обработано: {len(data.get('detections', []))} изображений")
    for d in data.get('detections', [])[:3]:
        print(f"  {d.get('image_ind','?'):<30} → {d.get('id_animal','?')} ({d.get('probability',0):.2f})")
else:
    print('Нет примеров для batch — пропускаем.')

## Что дальше

- **Web UI без Colab:** запустить `docker compose up --build` на вашем компьютере и открыть <http://localhost:8080>. Инструкция: `docs/USER_GUIDE_BIOLOGIST.md`.
- **Streamlit демо:** `cd research/demo-ui && streamlit run streamlit_app.py`.
- **CLI:** `python -m whales_identify predict /path/to/photo.jpg` — без сервера, прямой вызов модели.
- **API для своих интеграций:** HappyWhale / GBIF / iNaturalist коннекторы живут в `integrations/`. См. `DOCS/INTEGRATION_GUIDE.md`.
- **Дообучение на своих данных:** `docs/USER_TESTING_REPORT.md` и `whales_identify/train.py`.
- **Метрики production:** <http://localhost:8000/metrics> (Prometheus-совместимый формат).

Обратная связь и ошибки — GitHub Issues. Научные вопросы — email команды (см. `wiki_content/Contributing.md`).

## Shutdown

Останавливаем uvicorn при завершении работы в Colab.

In [ ]:
proc.terminate()
proc.wait(timeout=10)
print('✓ Backend остановлен')